# Titanic
## Score: .77990

In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from catboost import CatBoostClassifier
from pathlib import Path

_root = Path.cwd()
_data = _root / "titanic"

trainData = pd.read_csv(_data / "train.csv")
testData = pd.read_csv(_data / "test.csv")


def processAge(data):
    guess_ages = np.zeros((2, 3))
    for i in range(0, 1):
        for j in range(0, 3):
            guess_df = data[(data["Sex"] == i) & (data["Pclass"] == j + 1)]["Age"].dropna()
            age_guess = guess_df.median()
            guess_ages[i, j] = int(age_guess / 0.5 + 0.5) * 0.5

    for i in range(0, 2):
        for j in range(0, 3):
            data.loc[
                (data.Age.isnull()) & (data.Sex == i) & (data.Pclass == j + 1),
                "Age",
            ] = guess_ages[i, j]

    data["Age"] = data["Age"].astype(int)

    data.loc[data["Age"] <= 16, "Age"] = 0
    data.loc[(data["Age"] > 16) & (data["Age"] <= 32), "Age"] = 1
    data.loc[(data["Age"] > 32) & (data["Age"] <= 48), "Age"] = 2
    data.loc[(data["Age"] > 48) & (data["Age"] <= 64), "Age"] = 3
    data.loc[data["Age"] > 64, "Age"] = 4

    return data


def preprocessing(data):
    data = data.drop(columns=["Name", "PassengerId", "Ticket", "Cabin"])
    tsex = {"male": 0, "female": 1}
    tembarked = {"S": 0, "C": 1, "Q": 2}
    data.Sex = data.Sex.map(tsex)
    data.Embarked = data.Embarked.map(tembarked)

    data = processAge(data)

    for col in data.columns:
        data[col] = data[col] / data[col].abs().max()
    data.Fare = data.Fare * 1.8
    data = data.fillna(0)

    if "Survived" in data.columns:
        data = data.drop(columns="Survived")

    return data


trainLabel = trainData.Survived
female = trainData["Sex"] == "female"
child = trainData["Age"].notna() & (trainData["Age"] <= 16)
row_w = np.where(female, 3.0, 1.0).astype(np.float64)
row_w = np.where(child, row_w * 1.6, row_w)
row_w = row_w / row_w.mean()

processedTrainData = preprocessing(trainData)

X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    processedTrainData,
    trainLabel,
    row_w,
    test_size=0.1,
    random_state=42,
    stratify=trainLabel,
)

clf = CatBoostClassifier(
    bagging_temperature=0.2, metric_period=1000, learning_rate=0.0001
)
clf.fit(
    X_train,
    y_train,
    sample_weight=w_train,
    eval_set=(X_val, y_val),
    verbose=False,
)

prediction = clf.predict_proba(X_val)
rounded_predictions = np.argmax(prediction, axis=-1)
c_matrix = confusion_matrix(y_val, rounded_predictions)
dt_acc = c_matrix.trace() / c_matrix.sum()
print(c_matrix)
print(dt_acc)

clf.fit(
    processedTrainData,
    trainLabel,
    sample_weight=row_w,
    verbose=False,
)

testData = pd.read_csv(_data / "test.csv")
processedTestData = preprocessing(testData)
final_predictions = clf.predict(processedTestData)

output = pd.DataFrame(
    {"PassengerId": testData.PassengerId, "Survived": final_predictions}
)
output.to_csv(_root / "submission.csv", index=False)
output.head()

[[51  4]
 [14 21]]
0.8


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
